# Step 3.1 — LiDAR Multi-Frame Tracking (Upgraded, No UKF Yet) ✅

| | |
|---|---|
| **Input** | `output/step_2/lidar/<sample>/lidar_clusters.json` (Step 2.1), `output/step_1/lidar/<sample>/lidar_meta.json` (Step 1.3) |
| **Outputs** | `output/step_3/lidar/track_<id>.json` — one file per track, points in GLOBAL frame |
| | `output/step_4/lidar_tracking_summary.csv` |
| **Used by** | Step 5 (TTC estimation, LiDAR-only baseline), Step 4.4 (fusion) |

---

### Three real bugs fixed

1. **Ego-motion contamination.** Centroids were tracked in the LiDAR sensor's own local frame. Since the ego vehicle moves 4–7m between frames at typical urban speed, a parked car could appear to "move" far enough to break the 3.0m association threshold every single frame — fragmenting real objects into many short tracks. Fixed by transforming every centroid into the **global frame** (using Step 1.3's calibration) before tracking, so only true object motion affects matching.
2. **Double-assignment bug.** Each track independently grabbed its nearest detection with no mechanism to stop two tracks claiming the same detection. Fixed with the Hungarian algorithm (`scipy.optimize.linear_sum_assignment`) for proper one-to-one matching.
3. **No track termination.** Your dissertation describes tracks being dropped after a limited number of missed frames — the code didn't actually do this. Fixed with a `MAX_MISSED_FRAMES` eviction rule.

### Also removed

Unused radar/YOLO loading code (had the same `["points"]` unwrap bug as elsewhere, and was never actually used — `fused_frame = lidar_dets` only). This notebook is the LiDAR-only baseline by design; true fusion happens in Step 4.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import STEP0_DIR, STEP1_DIR, STEP2_DIR, STEP3_DIR

LIDAR_CLUSTERS_DIR = STEP2_DIR / "lidar"
LIDAR_META_DIR      = STEP1_DIR / "lidar"
FUSION_OUT_DIR       = STEP3_DIR / "lidar"
FUSION_OUT_DIR.mkdir(parents=True, exist_ok=True)

for p, name in [(LIDAR_CLUSTERS_DIR, "Step 2.1 output"), (LIDAR_META_DIR, "Step 1.3 output")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} not found at {p} — run that step first.")

print(f"✅ LIDAR_CLUSTERS_DIR: {LIDAR_CLUSTERS_DIR}")
print(f"✅ LIDAR_META_DIR    : {LIDAR_META_DIR}")
print(f"✅ FUSION_OUT_DIR    : {FUSION_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ LIDAR_CLUSTERS_DIR: F:\Sensor fusion Research\output\step_2\lidar
✅ LIDAR_META_DIR    : F:\Sensor fusion Research\output\step_1\lidar
✅ FUSION_OUT_DIR    : F:\Sensor fusion Research\output\step_3\lidar


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

ASSOC_DIST_THRESHOLD = 3.0   # metres — max distance to associate a detection with an existing track
MAX_MISSED_FRAMES    = 3     # track is dropped after this many consecutive frames with no match

print(f"✅ ASSOC_DIST_THRESHOLD = {ASSOC_DIST_THRESHOLD}m, MAX_MISSED_FRAMES = {MAX_MISSED_FRAMES}")

✅ ASSOC_DIST_THRESHOLD = 3.0m, MAX_MISSED_FRAMES = 3


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Sensor-to-global transform
# Moved to src/geometry.py — shared with Step 1.1, Step 2.3.1, Step 3.2
# ─────────────────────────────────────────────────────────────────

import numpy as np
from pyquaternion import Quaternion
from src.geometry import transform_matrix, point_to_global as centroid_to_global

print("\u2705 Transform utilities loaded.")


✅ Transform utilities loaded.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Tracker with proper one-to-one assignment + track eviction
# FIXED: Hungarian algorithm (was greedy per-track argmin — could double-assign)
# FIXED: tracks are dropped after MAX_MISSED_FRAMES (was: tracked forever)
# FIXED: scene-boundary isolation — tracks never associate across a scene_name
#        change; at a boundary all active tracks are force-finalized and a
#        fresh track table starts for the new scene (no carried IDs/state).
# ─────────────────────────────────────────────────────────────────

import uuid
from scipy.optimize import linear_sum_assignment


class GlobalFrameTracker:
    def __init__(self, dist_thresh, max_missed):
        self.active_tracks = {}     # tid -> {"points": [(sample_id, timestamp, [x,y,z])...], "missed": int}
        self.finished_tracks = {}   # tid -> same structure, evicted (kept for saving)
        self.dist_thresh = dist_thresh
        self.max_missed = max_missed
        self.current_scene_name = None
        self.association_log = []  # (prev_sample_id, prev_scene, curr_sample_id, curr_scene) per accepted match

    def update(self, detections_global, sample_id, timestamp, scene_name):
        """detections_global: list of [x, y, z] centroids already in global frame."""
        if self.current_scene_name is not None and scene_name != self.current_scene_name:
            # Scene boundary: close out every active track from the previous scene and
            # start a fresh, empty track table. No track IDs, motion state, or "last known
            # position" carry across — each survivor is just finalized as-is.
            self.finished_tracks.update(self.active_tracks)
            self.active_tracks = {}
        self.current_scene_name = scene_name

        det_arr = np.array(detections_global) if detections_global else np.empty((0, 3))
        track_ids = list(self.active_tracks.keys())
        n_tracks, n_dets = len(track_ids), len(det_arr)

        matched_track_idx, matched_det_idx = set(), set()

        if n_tracks > 0 and n_dets > 0:
            cost = np.zeros((n_tracks, n_dets))
            for i, tid in enumerate(track_ids):
                pts = self.active_tracks[tid]["points"]
                last_pos = np.array(pts[-1][2], dtype=float)
                pred = last_pos
                if len(pts) >= 2 and pts[-1][1] is not None and pts[-2][1] is not None:
                    dt_prev = (pts[-1][1] - pts[-2][1]) / 1e6
                    dt_now  = (timestamp   - pts[-1][1])  / 1e6
                    if dt_prev > 0 and dt_now > 0:
                        vel = (last_pos - np.array(pts[-2][2], dtype=float)) / dt_prev
                        spd = np.linalg.norm(vel)
                        MIN_TRUSTED_SPEED = 1.0     # m/s — below this, treat as sensor noise
                        if spd < MIN_TRUSTED_SPEED:
                            vel = np.zeros(3)        # NEW LINE — don't trust jitter as motion
                        elif spd > 30.0:
                            vel = vel / spd * 30.0
                        pred = last_pos + vel * dt_now
                cost[i] = np.linalg.norm(det_arr - pred, axis=1)
            row_idx, col_idx = linear_sum_assignment(cost)   # optimal one-to-one matching
            for r, c in zip(row_idx, col_idx):
                if cost[r, c] < self.dist_thresh:
                    tid = track_ids[r]
                    prev_sample_id, prev_timestamp, _ = self.active_tracks[tid]["points"][-1]
                    self.association_log.append((prev_sample_id, self.current_scene_name, sample_id, scene_name))
                    self.active_tracks[tid]["points"].append((sample_id, timestamp, detections_global[c]))
                    self.active_tracks[tid]["missed"] = 0
                    matched_track_idx.add(r)
                    matched_det_idx.add(c)

        # Unmatched existing tracks — increment missed count, evict if stale
        for i, tid in enumerate(track_ids):
            if i not in matched_track_idx:
                self.active_tracks[tid]["missed"] += 1
                if self.active_tracks[tid]["missed"] > self.max_missed:
                    self.finished_tracks[tid] = self.active_tracks.pop(tid)

        # Unmatched detections — start new tracks
        for j in range(n_dets):
            if j not in matched_det_idx:
                tid = str(uuid.uuid4())[:8]
                self.active_tracks[tid] = {
                    "points": [(sample_id, timestamp, detections_global[j])],
                    "missed": 0
                }

    def save_tracks(self, out_dir):
        all_tracks = {**self.finished_tracks, **self.active_tracks}
        n_saved = 0
        for tid, track in all_tracks.items():
            if len(track["points"]) >= 2:
                with open(out_dir / f"track_{tid}.json", "w") as f:
                    json.dump(track["points"], f, indent=2)
                n_saved += 1
        return n_saved, len(all_tracks)


print("GlobalFrameTracker defined.")


GlobalFrameTracker defined.


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Main loop: transform centroids to global frame, then track
# ─────────────────────────────────────────────────────────────────

import json
from tqdm import tqdm

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

tracker = GlobalFrameTracker(dist_thresh=ASSOC_DIST_THRESHOLD, max_missed=MAX_MISSED_FRAMES)
n_samples_processed = 0
n_samples_no_clusters = 0

for sample_id in tqdm(sorted(samples_index.keys()), desc="Tracking LiDAR objects"):
    clusters_path = LIDAR_CLUSTERS_DIR / sample_id / "lidar_clusters.json"
    meta_path = LIDAR_META_DIR / sample_id / "lidar_meta.json"

    if not clusters_path.exists() or not meta_path.exists():
        continue

    with open(clusters_path) as f:
        clusters = json.load(f)
    with open(meta_path) as f:
        lidar_meta = json.load(f)

    scene_name = samples_index[sample_id]["scene_name"]

    if len(clusters) == 0:
        n_samples_no_clusters += 1
        tracker.update([], sample_id, samples_index[sample_id]["timestamp_us"], scene_name)
        continue

    ego_pose = lidar_meta["ego_pose"]
    calib = {
        "translation": lidar_meta["sensor_to_ego_translation"],
        "rotation": lidar_meta["sensor_to_ego_rotation"]
    }

    # FIXED — transform every centroid to global frame BEFORE tracking
    detections_global = [
        centroid_to_global(v["centroid"], ego_pose, calib).tolist()
        for v in clusters.values()
    ]

    timestamp = samples_index[sample_id]["timestamp_us"]
    tracker.update(detections_global, sample_id, timestamp, scene_name)
    n_samples_processed += 1

n_saved, n_total = tracker.save_tracks(FUSION_OUT_DIR)

# ── Scene-boundary isolation check ─────────────────────────────────
cross_scene_associations = [a for a in tracker.association_log if a[1] != a[3]]
assert len(cross_scene_associations) == 0, \
    f"{len(cross_scene_associations)} cross-scene associations found: {cross_scene_associations[:5]}"

print(f"\nStep 4.1 complete.")
print(f"   Samples processed        : {n_samples_processed}")
print(f"   Samples with zero clusters: {n_samples_no_clusters}")
print(f"   Total tracks created      : {n_total}")
print(f"   Tracks saved (length >= 2) : {n_saved}")
print(f"   Associations logged        : {len(tracker.association_log)} (0 cross-scene, verified)")
print(f"Saved to: {FUSION_OUT_DIR}")


Tracking LiDAR objects:   0%|          | 0/404 [00:00<?, ?it/s]

Tracking LiDAR objects:   0%|          | 1/404 [00:00<05:18,  1.26it/s]

Tracking LiDAR objects:   1%|          | 3/404 [00:00<01:44,  3.84it/s]

Tracking LiDAR objects:   1%|          | 5/404 [00:01<01:01,  6.51it/s]

Tracking LiDAR objects:   2%|▏         | 7/404 [00:01<00:46,  8.53it/s]

...

Tracking LiDAR objects:  99%|█████████▉| 399/404 [00:33<00:00, 15.49it/s]

Tracking LiDAR objects:  99%|█████████▉| 401/404 [00:33<00:00, 14.32it/s]

Tracking LiDAR objects: 100%|█████████▉| 403/404 [00:34<00:00, 15.07it/s]

Tracking LiDAR objects: 100%|██████████| 404/404 [00:34<00:00, 11.83it/s]


Step 4.1 complete.
   Samples processed        : 404
   Samples with zero clusters: 0
   Total tracks created      : 4778
   Tracks saved (length >= 2) : 3152
   Associations logged        : 12983 (0 cross-scene, verified)
Saved to: F:\Sensor fusion Research\output\step_3\lidar


In [6]:
"""
diagnose_step1_regression.py

You ran the Step 1 patch and track count got WORSE (4513 -> 9484), not better.
This script tells you which of two things happened:

  (A) IMPLEMENTATION BUG — the prediction math itself is wrong (sign error, unit
      mismatch, wrong index). Fixable in 5 minutes once we see the numbers.

  (B) NOISE AMPLIFICATION — the math is correct, but most LiDAR clusters are
      near-static clutter (kerbs, poles, building fragments), and using 2 noisy
      points to estimate velocity creates fake motion that pushes the predicted
      position AWAY from the real (mostly still) next detection. This means
      Phase 2 (filtering clutter) needs to happen BEFORE Phase 1, not after.

Run this INSIDE Step_3_1_LiDAR_Fusion, right after your Cell 4 (the
patched tracker) and Cell 5 (the main loop) have both run — so `tracker` still
holds real data. Paste this as a new cell at the end and run it.
"""
import numpy as np

# ── Recompute, for every association attempt, whether the prediction helped ──
# We re-run the same loop logic used inside update(), but this time we log
# BOTH the "old way" distance (from last_pos) and the "new way" distance
# (from pred) to whatever the tracker actually matched, so we can see which
# one was closer to truth more often.

improved, worsened, unchanged = 0, 0, 0
deltas = []          # positive = prediction moved AWAY from the eventual match (bad)
speeds = []           # the estimated speed for every 2+-point track, at time of prediction

for tid, track in {**tracker.finished_tracks, **tracker.active_tracks}.items():
    pts = track["points"]
    for k in range(2, len(pts)):
        prev2, prev1, cur = pts[k-2], pts[k-1], pts[k]
        t_prev2, pos_prev2 = prev2[1], np.array(prev2[2], dtype=float)
        t_prev1, pos_prev1 = prev1[1], np.array(prev1[2], dtype=float)
        t_cur,   pos_cur   = cur[1],   np.array(cur[2],   dtype=float)

        dt_prev = (t_prev1 - t_prev2) / 1e6
        dt_now  = (t_cur   - t_prev1) / 1e6
        if dt_prev <= 0 or dt_now <= 0:
            continue

        vel = (pos_prev1 - pos_prev2) / dt_prev
        spd = np.linalg.norm(vel)
        speeds.append(spd)
        if spd > 30.0:
            vel = vel / spd * 30.0
        pred = pos_prev1 + vel * dt_now

        dist_old = np.linalg.norm(pos_cur - pos_prev1)   # "last position" method
        dist_new = np.linalg.norm(pos_cur - pred)         # "predicted position" method
        delta = dist_new - dist_old                       # negative = prediction helped
        deltas.append(delta)

        if delta < -0.05:
            improved += 1
        elif delta > 0.05:
            worsened += 1
        else:
            unchanged += 1

deltas = np.array(deltas)
speeds = np.array(speeds)

print("=" * 70)
print("DIAGNOSTIC: did the motion prediction help or hurt, on your actual data?")
print("=" * 70)
print(f"Association steps analysed : {len(deltas)}")
print(f"Prediction MOVED CLOSER to next real detection : {improved} ({improved/max(len(deltas),1)*100:.1f}%)")
print(f"Prediction MOVED FARTHER                        : {worsened} ({worsened/max(len(deltas),1)*100:.1f}%)")
print(f"No meaningful difference                         : {unchanged} ({unchanged/max(len(deltas),1)*100:.1f}%)")
print()
print(f"Median delta (negative = prediction helped) : {np.median(deltas):+.3f} m")
print(f"Mean   delta                                 : {np.mean(deltas):+.3f} m")
print()
print("Estimated speed distribution (this is the 'velocity' the code computed):")
print(f"  median {np.median(speeds):.2f} m/s   mean {np.mean(speeds):.2f} m/s   "
      f"90th pct {np.percentile(speeds, 90):.2f} m/s   max {speeds.max():.2f} m/s")
print(f"  fraction of estimated speeds ABOVE 15 m/s (54 km/h, implausible for most objects) "
      f": {(speeds > 15).mean()*100:.1f}%")
print()

# ── Verdict ──
if worsened > improved * 1.2:
    print(">>> VERDICT: NOISE AMPLIFICATION (hypothesis B).")
    print("    The prediction is making the match WORSE more often than it helps.")
    print("    This is consistent with velocity being estimated from noisy 2-point")
    print("    differences on mostly-static clutter, not a code bug.")
    print("    NEXT STEP: do Phase 2 (LiDAR extent filter, radar clustering) FIRST,")
    print("    then re-apply Phase 1's motion prediction on the cleaned data.")
elif improved > worsened * 1.2:
    print(">>> VERDICT: prediction IS helping at the point-pair level.")
    print("    If track count still went up despite this, the regression is likely")
    print("    coming from somewhere else — check the MAX_MISSED_FRAMES eviction")
    print("    logic, or whether `timestamp` is arriving as expected. Print the")
    print("    first 5 calls to `.update()` in Cell 5 and manually inspect the")
    print("    `timestamp` and `dt_now` values.")
else:
    print(">>> VERDICT: prediction is roughly a wash at the point-pair level —")
    print("    not clearly better or worse. Track count doubling then likely comes")
    print("    from a DIFFERENT source. Check for a duplicate-run artifact: did you")
    print("    re-run Cell 5 twice without restarting the kernel? Old `tracker` state")
    print("    can persist and double-count. Confirm with a fresh kernel restart.")

print()
print("If speeds above are mostly under 2-3 m/s with occasional huge outliers,")
print("that ALSO points to noise amplification: real objects don't have 20 m/s")
print("bursts between consecutive frames unless the underlying detection jumped.")

DIAGNOSTIC: did the motion prediction help or hurt, on your actual data?
Association steps analysed : 9831
Prediction MOVED CLOSER to next real detection : 2871 (29.2%)
Prediction MOVED FARTHER                        : 4910 (49.9%)
No meaningful difference                         : 2050 (20.9%)



Median delta (negative = prediction helped) : +0.050 m
Mean   delta                                 : -0.056 m

Estimated speed distribution (this is the 'velocity' the code computed):


  median 0.65 m/s   mean 1.18 m/s   90th pct 2.98 m/s   max 13.96 m/s
  fraction of estimated speeds ABOVE 15 m/s (54 km/h, implausible for most objects) : 0.0%

>>> VERDICT: NOISE AMPLIFICATION (hypothesis B).
    The prediction is making the match WORSE more often than it helps.
    This is consistent with velocity being estimated from noisy 2-point
    differences on mostly-static clutter, not a code bug.
    NEXT STEP: do Phase 2 (LiDAR extent filter, radar clustering) FIRST,
    then re-apply Phase 1's motion prediction on the cleaned data.

If speeds above are mostly under 2-3 m/s with occasional huge outliers,
that ALSO points to noise amplification: real objects don't have 20 m/s
bursts between consecutive frames unless the underlying detection jumped.


In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Track length distribution — sanity check for fragmentation
# If this fix worked, you should see noticeably fewer 2-frame tracks
# and more longer tracks compared to the pre-fix version.
# ─────────────────────────────────────────────────────────────────

import pandas as pd

track_lengths = []
for track_file in FUSION_OUT_DIR.glob("track_*.json"):
    with open(track_file) as f:
        track = json.load(f)
    track_lengths.append(len(track))

length_df = pd.DataFrame({"track_length": track_lengths})
summary_path = STEP3_DIR / "lidar_tracking_summary.csv"
length_df.to_csv(summary_path, index=False)

print(f"✅ Summary saved: {summary_path}")
print(f"   Total tracks       : {len(length_df)}")
print(f"   Mean track length  : {length_df['track_length'].mean():.1f} frames")
print(f"   Median track length: {length_df['track_length'].median():.0f} frames")
print(f"   Tracks of length 2 (minimum, most fragmented): "
      f"{(length_df['track_length'] == 2).sum()} ({(length_df['track_length'] == 2).mean()*100:.1f}%)")
print(f"   Tracks of length 10+: {(length_df['track_length'] >= 10).sum()}")

display(length_df.describe())

✅ Summary saved: F:\Sensor fusion Research\output\step_3\lidar_tracking_summary.csv
   Total tracks       : 3152
   Mean track length  : 5.1 frames
   Median track length: 3 frames
   Tracks of length 2 (minimum, most fragmented): 1042 (33.1%)
   Tracks of length 10+: 342


,track_length
count,3152.000000
mean,5.118972
std,5.372698
min,2.000000
25%,2.000000
50%,3.000000
75%,6.000000
max,41.000000
